In [20]:
import pandas as pd

In [36]:
def update_time_and_addeddate(df: pd.DataFrame, time_col: str = "time_", date_col: str = "addeddate", output_format: str = "%Y-%m-%d %H:%M:%S"):

    # Convert to datetime; invalid parses become NaT
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

    # Drop rows where either column is null/invalid
    df = df.dropna(subset=[time_col, date_col])

    # Format both columns to the same string format
    df[time_col] = df[time_col].dt.strftime(output_format)
    df[date_col] = df[date_col].dt.strftime(output_format)

    return df


In [47]:
#Convert Addeddate and Time_ to datetime
def convertToDatetime(df : pd.DataFrame):
    df["time_"] = pd.to_datetime(df["time_"], format="%I:%M%p").dt.strftime("%H:%M:%S")
    df["addeddate"] = pd.to_datetime(df["addeddate"], errors="coerce").dt.strftime("%d/%m/%Y")
    df["timestamp"] = pd.to_datetime(df["addeddate"] + " " + df["time_"], format="%d/%m/%Y %H:%M:%S")
    df.drop(["addeddate", "time_"], axis=1, inplace=True)
    
    return df

In [48]:
def convertToNumeric(df: pd.DataFrame):
    for col in ["Sheet", "Sales_Sheet", "Sales_pack", "zone_id", "pharmacy_id"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


In [49]:
def create_temporal_features(df: pd.DataFrame):
    df["day_of_week"] = df["timestamp"].dt.dayofweek
    df["month"] = df["timestamp"].dt.month
    df["week_of_year"] = df["timestamp"].dt.isocalendar().week.astype(int)
    df["day"] = df["timestamp"].dt.day
    df["hour"] = df["timestamp"].dt.hour
    df["weekend"] = (df["day_of_week"].isin([5, 6]).astype(int))

    return df

In [50]:
def preprocess_data(df: pd.DataFrame, cityNum:int, zoneNum:int):
    df = convertToDatetime(df)
    df = convertToNumeric(df)
    df = create_temporal_features(df)

    df.to_csv(f"../PharmacyTransactionalDataset\\PreprocessedData\\C{cityNum}_Z{zoneNum}.csv", index=False)

In [51]:
for city_num in range(1, 4):
    for zone_num in range(1, 4):
        file_path = f"../PharmacyTransactionalDataset\\City{city_num}\\C{city_num}_Z{zone_num}.csv"
        df = pd.read_csv(file_path)
        preprocess_data(df, city_num, zone_num)

ValueError: cannot convert NA to integer